# Practice 102 — Panel Data & Fixed Effects

**Theoretical context**: see `CLAUDE.md` in this folder before starting.

**Phases**: this notebook mirrors the phases in `CLAUDE.md` § Instructions.
Each phase's exercise calls into a `src/_0N_<phase_name>.py` companion module —
read that module's `TODO(human)` block before implementing it there, then
re-run the corresponding cell below.

## Setup

In [ ]:
import sys
from pathlib import Path

# Jupyter sets the kernel's cwd to this notebook's folder, so the practice root --
# where the `src` package lives -- is not on sys.path. Put it there.
_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import time

import numpy as np
import pandas as pd
import xy.pyplot as plt

from src.datasets import SMALL_PANEL, LARGE_PANEL, generate_panel
from src.plotting import coefficient_comparison_plot, coverage_plot

## Phase 1 — The within (demeaning) transformation

We simulate a panel with a **known** true beta on `[x1, x2]` (see `src/datasets.py`).
`alpha_i` (the unit effect) is built to correlate with `x1`'s unit mean — the classic
omitted-variable-bias setup for fixed effects. Pooled OLS below has no way to see
`alpha_i`; watch its `beta_x1` come out biased away from the truth.

In [ ]:
panel = generate_panel(**SMALL_PANEL, seed=0)
df = panel.df
print(f"True beta: {panel.beta_true}")
df.head()

In [ ]:
from src._01_within_transform import ols_lstsq

X_pooled = df[["x1", "x2"]].to_numpy()
y = df["y"].to_numpy()
pooled_fit = ols_lstsq(X_pooled, y)
print(f"Pooled OLS beta:  {pooled_fit.beta}")
print(f"True beta:        {panel.beta_true}")

### Exercise — `src/_01_within_transform.py :: within_demean`

Open `src/_01_within_transform.py`, read the `TODO(human)` block above the
function, implement it there (not in this cell), then re-run the cell below.

In [ ]:
from src._01_within_transform import fit_within, validate_against_libraries, within_demean

validate_against_libraries(df)
within_fit = fit_within(df, "y", ["x1", "x2"], "unit_id")
y_tilde, X_tilde = within_demean(y, X_pooled, df["unit_id"].to_numpy())
print(f"Within (one-way FE) beta: {within_fit.beta}")

## Phase 2 — Two-way fixed effects

`gamma_t` (the time effect) trends upward together with `x2`'s rollout, so
one-way FE is still biased for `beta_x2` — check `within_fit.beta[1]` against
the true `beta_x2` above; it's closer than pooled OLS but not exact. Two-way
FE removes the time trend too.

### Exercise — `src/_02_two_way_demean.py :: two_way_demean`

Open `src/_02_two_way_demean.py`, read the `TODO(human)` block above the
function, implement it there, then re-run the cell below.

In [ ]:
from src._02_two_way_demean import fit_two_way, two_way_demean
from src._02_two_way_demean import validate_against_libraries as validate_two_way

validate_two_way(df)
two_way_fit = fit_two_way(df, "y", ["x1", "x2"], "unit_id", "time_id")
y_tilde_tw, X_tilde_tw = two_way_demean(y, X_pooled, df["unit_id"].to_numpy(), df["time_id"].to_numpy())
print(f"Two-way FE beta: {two_way_fit.beta}")
print(f"True beta:       {panel.beta_true}")

## Phase 3 — Frisch-Waugh-Lovell: why demeaning works

Demeaning is an application of the FWL theorem with `W` set to a full set of
unit dummy columns. This phase implements the *general* partialling-out
routine and checks it reproduces Phase 1's within-estimator beta exactly —
without ever demeaning anything.

### Exercise — `src/_03_fwl_demonstration.py :: fwl_partial_out`

Open `src/_03_fwl_demonstration.py`, read the `TODO(human)` block above the
function, implement it there, then re-run the cell below.

In [ ]:
from src._03_fwl_demonstration import demonstrate_equivalence

demonstrate_equivalence(df)

## Phase 4 — Clustering at the level of treatment assignment

`x2` is assigned at the cluster-period level and the error term carries a
matching cluster-period shock. Naive standard errors treat every row as
independent information; they aren't. We compare naive vs. cluster-robust
standard errors on one sample, then run a Monte Carlo coverage simulation.

### Exercise — `src/_04_cluster_vcov.py :: cluster_robust_vcov`

Open `src/_04_cluster_vcov.py`, read the `TODO(human)` block above the
function, implement it there, then re-run the cell below.

In [ ]:
from src._04_cluster_vcov import naive_vcov, cluster_robust_vcov, simulate_cluster_coverage

naive_se = np.sqrt(np.diag(naive_vcov(X_tilde, within_fit.resid, within_fit.XtX_inv)))
cluster_se = np.sqrt(np.diag(cluster_robust_vcov(
    X_tilde, within_fit.resid, within_fit.XtX_inv, df["cluster_id"].to_numpy()
)))
print(f"Naive SE:   {naive_se}")
print(f"Cluster SE: {cluster_se}")

In [ ]:
print("Running coverage simulation (a couple hundred re-fits, a few seconds)...")
coverage = simulate_cluster_coverage(n_reps=200)
print(f"Coverage - naive:     {coverage['naive']:.2%}")
print(f"Coverage - clustered: {coverage['clustered']:.2%}")

## Phase 5 — End-to-end: coefficient plot, coverage plot, speed benchmark

Same three point estimates (pooled OLS, one-way within, two-way FE) against
the truth, then the coverage simulation as a plot, then a speed comparison
between the hand-rolled within estimator, `pyfixest`, and
`linearmodels.PanelOLS` on a much larger panel — this is where `pyfixest`'s
`reghdfe`-style absorption earns its place.

In [ ]:
est_by_method = {
    "pooled OLS": pooled_fit.beta,
    "within (one-way FE)": within_fit.beta,
    "two-way FE": two_way_fit.beta,
}
se_by_method = {
    "pooled OLS": np.sqrt(np.diag(naive_vcov(X_pooled, pooled_fit.resid, pooled_fit.XtX_inv))),
    "within (one-way FE)": np.sqrt(np.diag(naive_vcov(X_tilde, within_fit.resid, within_fit.XtX_inv))),
    "two-way FE": np.sqrt(np.diag(naive_vcov(X_tilde_tw, two_way_fit.resid, two_way_fit.XtX_inv))),
}
fig = coefficient_comparison_plot(["x1", "x2"], est_by_method, se_by_method, panel.beta_true)
fig

In [ ]:
fig2 = coverage_plot(coverage)
fig2

In [ ]:
import pyfixest as pf
from linearmodels.panel import PanelOLS

large_panel = generate_panel(**LARGE_PANEL, seed=0)
large_df = large_panel.df
large_panel_df = large_df.set_index(["unit_id", "time_id"])

t0 = time.perf_counter()
fit_within(large_df, "y", ["x1", "x2"], "unit_id")
t_ours = time.perf_counter() - t0

t0 = time.perf_counter()
pf.feols("y ~ x1 + x2 | unit_id", data=large_df)
t_pyfixest = time.perf_counter() - t0

t0 = time.perf_counter()
PanelOLS(large_panel_df["y"], large_panel_df[["x1", "x2"]], entity_effects=True).fit()
t_linearmodels = time.perf_counter() - t0

print(f"rows: {len(large_df):,}")
print(f"hand-rolled within: {t_ours:.3f}s")
print(f"pyfixest:           {t_pyfixest:.3f}s")
print(f"linearmodels:       {t_linearmodels:.3f}s")

## Verification

Sanity-checks that must pass once every TODO is implemented.

In [ ]:
assert np.allclose(within_fit.beta, panel.beta_true, atol=0.5), "within beta should be close to the true beta"
assert np.allclose(two_way_fit.beta, panel.beta_true, atol=0.3), "two-way FE beta should be closer still"
assert np.abs(pooled_fit.beta[0] - panel.beta_true[0]) > np.abs(within_fit.beta[0] - panel.beta_true[0]), (
    "pooled OLS should be more biased on beta_x1 than the within estimator"
)
assert coverage["clustered"] > coverage["naive"], "clustered CIs should cover the truth more often than naive ones"
print("OK")